# __ML in der Logistik__
# Exercise 01 - Get started with machine learning and Python

__Goal__: Understand how to visualize the data, preprocess the data, apply a ML algorithm, and measure the performance.


__Data and application__: The dataset used contains 204 AIS messages recorded for 10 trips on the route Rotterdam > Hamburg. We want to build a ML model to predict the estimated time of arrival of a ship.



### Contents

* [0 Use of Jupyter Notebook](#t0)
* [1 Load and visualize the data (10 min)](#t1)
* [2 Prepare the dataset](#t2)
    * [2.1 Attribute type (10 min)](#t21)   
    * [2.2 Create a feature (5 min)](#t22)  
    * [2.3 Wrong and missing values (15 min)](#t23)
* [3 Build a ML model](#t3)
    * [3.1 Select the attributes (10 min)](#t31)
    * [3.2 Split train-test (10 min)](#t32)
    * [3.3 Dummies (10 min)](#t33)
    * [3.4 Apply the algorithm (5 min)](#t34)
* [4 Measure the performance (5 min)](#t4)
* [5 Summary questions](#t5)

## 0 Use of Jupyter Notebook <a class="anchor" id="t0"></a>

Jupyter Notebook is a tool that allows an interactive exercise. The instructions are scattered with executable code lines, to be able to instantly practice the tools learned.

To execute a code cell, simply press ```Ctrl``` + ```Enter``` (```Strg``` + ```Enter```).

## 1 Load and visualize the data (10 min) <a class="anchor" id="t1"></a>

Firstly, we need to import the dataset that we will be using along the exercise.

To load a csv file, we can use the function ```read_csv()``` from the ```pandas``` library.

In [ ]:
import pandas as pd

df = pd.read_csv('01-preprocessing.csv')

We can take a look at the dataset with the method ```info()```. Also, we can print the first 5 instances with the method ```head()```.

The method ```describe()``` prints basic statistics of the attributes in the dataset (count, mean, std, distribution, ...). Not all of them are important for you. Read the documentation for more information on the statistics.

The argument ```dtypes``` returns the type of each attribute in the dataset (the reader of the csv file converted automatically each numerical and string types into the right format).

In [ ]:
# Global visualization of the dataset
df.info()

In [ ]:
# Print first 5 instances
df.head()

In [ ]:
# Print basic statistics
df.describe()

In [ ]:
# Print types of attributes
df.dtypes

There are many different methods to graphically visualize the dataset. Here, we will use a few functions of the ```pandas``` library.

We can visualize the distribution of a numerical attribute with a histogram: ```hist()```.

In [ ]:
# Distribution of the Draught attribute
df['Draught'].plot.hist()

For a categorical attribute, we plot a bar chart: ```bar()```. For the purpose of the example, we converted the attribute TripID into a categorical attribute. The attributes types will be covered later in the exercise.

In [ ]:
# Convert the TripID attribute
example = df['TripID'].astype('category')
example.value_counts().sort_index().plot.bar()

## 2 Prepare the dataset <a class="anchor" id="t2"></a>

### 2.1 Attribute type (10 min) <a class="anchor" id="t21"></a>

Let's take a look at each attribute in the dataset and their types. In our case, the types can be:
+ ```int64``` and ```float64```: numeric
+ ```category```: categorical
+ ```object```: string
+ ```datetime64[ns]```: time stamp
+ ```timedelta64[ns]```: time difference

In [ ]:
# Types of attributes
df.dtypes

The ```object``` attributes can't be processed by the classic machine learning algorithms that we will be using. We need to convert them into a processable type.

The method ```astype('category')``` changes the type to categorical, while the function ```to_datetime()``` of the ```pandas``` library transforms the attribute to a timestamp.

__Change the type of each of these attributes (2 points).__

In [ ]:
# Change attributes type
df['StartTime'] = pd.to_datetime(df['StartTime'])

df['Name'] = df['Name'].astype('category')

However, there are 4 other numeric attributes that should be categorical. They can be processed as numeric by a machine learning algorithm, but they represent a variable that is not continuous and it makes more sense to treat them as a category.

__Identify the 4 numeric attributes that should be categorical and change their type (2 points).__

In [ ]:
# Change the type of 4 numeric attributes



### 2.2 Create a feature (5 min) <a class="anchor" id="t22"></a>

This dataset contains the information from AIS messages received for 8 trips on the route Rotterdam > Hamburg.

We want to predict the ETT of a trip. It is defined, from each position (each AIS message), as the remaining travel time before arrival. As you can see, the attribute ETT doesn't exist in the dataset: we have to create it. The ETT can be calculated from the existing attributes.

Each instance in the dataset represents one AIS message sent at a certain point of the trip.

__Propose a simple relation to calculate the ETT by combining the existing attributes of the dataset, and create the "ETT" attribute (2 points).__

In [ ]:
# Create the ETT attribute



Verify that the values of the attribute you created are consistent: the duration of the longest trip of the dataset is about 13 hours.

You can use the function ```describe()``` to print the statistics of the attributes.

In [ ]:
# Visually verify that it is consistent
df.describe()

To make it easier for the prediction later on, we want to convert the new attribute ``ETT`` into an numerical attribute, representing the time expressed in minutes. First, let's understand how the value that we created is represented by looking at its type:

In [ ]:
df['ETT'].dtype

If you created the attribute properly, it should be expressed in nanoseconds. We can simply convert it in seconds by using the function ``to_numeric()`` of the ``Pandas`` library and the appropriate calculation on the value. Let's create a new attribute ``ETT_minutes`` to have this change. This new attribute will be the one used for prediction in the rest of the exercise.

In [ ]:
df['ETT_minutes'] = pd.to_numeric(df['ETT'])/60000000000

We can verify that the attribute was properly created and that its value is correct.

In [ ]:
df.describe()

### 2.3 Wrong and missing values (15 min) <a class="anchor" id="t23"></a>

With the two ways of visualizing attributes we saw in section 1, we take a close look at each attribute of the dataset. That way, we can discover some distributions which seem weird and are probably noise or mistakes in recording.

In [ ]:
df.info()

In [ ]:
# Visualize the attributes one by one: categorical and time attributes

df['TripID'].value_counts().sort_index().plot.bar()
# change 'TripID' with the other attributes to visualize

In [ ]:
# Visualize the attributes one by one: numeric attributes

df['StartLatitude'].plot.hist()
# change 'StartLatitude' with the other attributes to visualize

We can also visualize the ``ETT`` attribute as we converted it into a numerical value.

In [ ]:
# Plot numeric histogram

df['ETT_minutes'].plot.hist()

From the visualization, we can identify 3 attributes which have a few samples with a wrong value.

To deal with instances with wrong or missing values, there are 3 possibilities:
+ we can recover the real values by guessing or predicting
+ we can set up a neutral value or class, corresponding to the missing value
+ we can delete the instances or the attribute

In our case, it might be possible to recover the values if the same trip contains other instances with a correct value. To verify that, we will first find the Trip ID of the concerned trip.

In [ ]:
# Find TripID of the trip with wrong values

df.loc[df['Length'] == 0]

We can do the same with the other 2 concerned attributes.

With the attribute ```index```, we get the index of the datapoints concerned for each attribute. Then, we have a list of the datapoints to modify. Also, we can compare if these are the same datapoints for the 3 wrongly recorded attributes.

In [ ]:
df.loc[df['Length'] == 0].index

From there, we can modify the value for these wrong datapoints by taking the right value for the same trip, if it exists. We see that the values which are wrong for the attribute 'Length' all belong to the trip with the TripID = 21113. Let's print the other values recorded for the 'Length' on this trip.

In [ ]:
df.loc[df['TripID'] == 21113, 'Length']

Finally, we can modify the value for the wrong points.

In [ ]:
df.loc[df['Length'] == 0, 'Length'] = 269

Now we can do the same with the other 2 attributes.

__Change the wrong values for the 2 other attributes (2 points).__

In [ ]:
# Change the wrong values for the 2 other attributes



## 3 Build a ML model <a class="anchor" id="t3"></a>

### 3.1 Select the attributes (10 min) <a class="anchor" id="t31"></a>

Before applying the ML algorithm, we need to specify the attributes we want to use for prediction. The list ``x`` will contain the names of the predictive attributes (used for prediction) and the list ``y`` contains the predicted attribute.

Here, we will start with a list ``x`` containing all the attributes, and we will remove one by one the ones we think are not useful for our prediction. Because we want to build a model that would be usable on the go, we remove the 'end' attributes, because in real time we do not know these values. Also, we choose to ignore the time attributes, because they cannot be used with our algorithm in this exercise.

Note: in other tasks, it is usually easier to just add the attributes we want to use. In this exercise, we want to make sure that we consider all the cases that we can encounter.

In [ ]:
# Initialize the lists of predictive and predicted attributes

x = ['TripID', 'MMSI', 'StartLatitude', 'StartLongitude', 'ID', 'shiptype', 'Length', 'Breadth',
    'Draught', 'Latitude', 'Longitude', 'SOG', 'COG', 'TH', 'Name', 'Callsign', 'AisSourcen']

y = ['ETT_minutes']

In machine learning, useless attributes are usually removed for the prediction, to decrease the complexity of the model and avoid some noise. These are:
+ categorical attributes with high uniquess
+ attributes with low variance
+ categorical attributes with the same distribution (identical)
+ highly correlated attributes

We can check the uniqueness of the attributes with the function ```nunique()```. Note: the function ```nunique()``` returns the number of unique instances. We divide it with the total number of instance to get a percentage of unique instances, easier to process.

In [ ]:
# Check uniqueness - only for categorical attributes

df.nunique().apply(lambda x: x/len(df))

We can identify a categorical attribute with a uniqueness of 1: it means that it is changing for each instance. Such an attribute is then useless for prediction, we can remove it.

__Remove the attribute with high uniqueness and set the variable__ ``t31_unique`` __with its name (string) (1 point).__

In [ ]:
# Remove the attribute with high uniqueness (replace the ? with the name of the attribute)

x.remove('?')

t31_unique = '?'

Now, we can also search for categorical attributes that have the same value for all AIS messages

In [ ]:
# Check whether a categorical attribute has the same value for all messages
df.nunique()

We can see that there is one attribute which doesn't vary: we can remove it.

__Remove the attribute which doesn't vary and set the variable__ ``t31_allSame`` __with its name (string) (1 point).__

In [ ]:
# Remove the attribute with low variance (replace the ? with the name of the attribute)

x.remove('?')

t31_allSame = '?'

If we take a look again at the distribution of the attributes (from the visualization in the part 1), we can identify a few categorical attributes which have the same distribution (i.e. they are not independent from each other).

In order to verify that 2 attributes are indeed dependent, we can print the cross table.

In [ ]:
# Check distribution of categorical attributes

pd.crosstab(df['MMSI'], df['Name'])

For instance, we can see that the attributes 'MMSI' and 'Name' have the exact same distribution (i.e. an instance with a value x1 for 'MMSI' will always have the value x2 for 'Name'): more than independent, they are identical and we can remove one to decrease the complexity of the model.

Note: two attributes can have the same distribution in the train dataset by chance, and have a different distribution in the rest of the data. When it is possible, we should always ensure that the action we do on the dataset makes sense. For instance here, it is not surprising that the attributes 'MMSI' and 'Name' are identical, as they both represent the ship used for the trip. It is important to always keep in mind the meaning of the variables we are handling, when possible.

There is another attribute which have the same distribution as 'Name' and 'MMSI'.

__Find this attribute by printing the crosstab, and remove 2 out of these 3 attributes (here, keep the 'Name' because we will be using it in the next part).__
__Set the variables__ ``t31_dis1`` __and__ ``t31_dis2`` __with the names of the 2 removed attributes (strings) (2 points).__

In [ ]:
# Remove the 2 attributes with the same distribution as MMSI

x.remove('?')
x.remove('?')

t31_dis1 = '?'
t31_dis2 = '?'

### 3.2 Split train-test (10 min) <a class="anchor" id="t32"></a>

To estimate the performance of a model, the ML model is applied on a new dataset which has not been used to build it, and the number of correctly classified instances is measured. As we usually get all the data in one dataset, we need to split this dataset into a training set and a testing set, so we can build the model on the training set and keep the testing set for measuring the performance.

Several methods can be used as a test option:
* __Separated training and testing sets__: from the available data, two datasets are created containing different instances. It can be used if the data have to be separated in a controlled way (for example: the AIS data have to be separated so that the training and testing sets contain AIS messages from different trips)
* __Percentage split__: the dataset can simply be randomly splitted and the model will be built on only one part of the available data. The value of the percentage is to be set up personally depending on the size of the dataset, it is common to use 80 - 20, or also 2/3 - 1/3.
* __Cross-validation__: the dataset is splitted into 10 subsets. The model is built on 9 subsets and tested on the 10th one. The operation is repeated until each dataset has been used for testing, and the final performance is the average on the 10 performances measured. This method of validation is often used on small datasets, when the data is precious.

As mentionned in the example, in our case we need to split the data according to the 'TripID' attribute, so that there is no datapoints in the training and the testing set that belong to the same trip.

First, we get a list of the different trips that we have with the method ```unique()```.

In [ ]:
ids = df['TripID'].unique()
print(ids)

From this list, we will randomly select 80% of the trips (8 trips out of 10).

In any ML program, we might need to use a random function. For example, here, we create a mask which randomly selects a percentage of the data. Sometimes, a ML algorithm can start with random parameters. For this reason, we can set up a random seed: that way, the results of the random function will be reproducible for later cases. Here, we use the function ``seed`` from the ``numpy`` library. In this exercise, we set up the value of the seed to 1, if we change this value, the randomization will be different.

In [ ]:
# Set up the random seed

import numpy as np

np.random.seed(1)

In [ ]:
# Randomly select 8 ids in the list

id_train = np.random.choice(ids, 8, False) # the parameter False allows for no duplicate in the selected ids

Finally, we can process the dataset. We create 2 new datasets ``train`` and ``test`` containing each the instances of the right TripID. For this we use the function ``isin()`` to get the values that belong to a list.

Optionally, we can make a copy of the instances with the function ``copy()``, so the initial dataset is not changed by further modifications that we could make on ``train`` and ``test``.

In [ ]:
# Create train and test sets

train = df.loc[df['TripID'].isin(id_train)].copy()
test = df.loc[~df['TripID'].isin(id_train)].copy()

We can always verify that the datasets we created make sense with the basic methods to analyse a dataset (from Part 1).

### 3.3 Dummies (10 min) <a class="anchor" id="t33"></a>

Some algorithms can only process numerical attributes. This is the case of the KNN algorithm that we want to use in this exercise.

Therefore, we need to transform the categorical attributes into numeric ones. For that, we will create one new attribute for each different class of the categorical attribute. Then, we can code the value of each class by adding the value 1 if the class is the corresponding one, or 0 if it is not. This method is called __one-hot encoding__ and the attributes created are called __dummy variables__ (dummies).

Here is an example of the method applied on the attribute 'Name':

![text](01-dummies.PNG)

The function ``get_dummies()`` from the ``Pandas`` library creates the dummies for a dataset passed in argument. Here is an example with the attribute 'Name'.

In [ ]:
# Get the dummies

dummies = pd.get_dummies(train['Name'], drop_first = False)
print(dummies)

In our case, we want to add a prefix to the name of the dummy variables, to identify the attribute that have been replaced.

In [ ]:
# Add the name of the attribute

dummies = dummies.add_prefix("Name#")
print(dummies)

Now that we have created the dummies for the attribute 'Name', we can replace it in the dataset. First, we drop the original column for the attribute 'Name', and then we join the created dummies.

In [ ]:
# Replace the attribute in the dataset

# Drop the old attribute
train.drop('Name', axis = 1, inplace = True)

# Add the dummies
train = train.join(dummies)

We can then check if the change was successful with the function ``info()``.

In [ ]:
train.info()

One last step: if the attribute we changed was in the list ``x``, we should change its name in the list.

In [ ]:
# Change the list x

# Remove the old attribute
x.remove('Name')

# Add the new attributes
x = [*x, *dummies.columns.values]

We can verify that the change has been successful.

In [ ]:
x

We can create a function to automatize these steps. Our function ``dummies()`` take as an argument the dataset, the list of predictive attributes ``x`` and the name of the attribute we want to modify, and returns the new dataset and the new list of predictive attributes.

In [ ]:
# Create function dummies()

def dummies(df, x, att_name):
    df_enc = df.copy()
    new_x = x.copy()
    
    dummies = pd.get_dummies(df[att_name], drop_first = False)
    dummies = dummies.add_prefix("{}#".format(att_name))
    df_enc.drop(att_name, axis = 1, inplace = True)
    df_enc = df_enc.join(dummies)
    
    if att_name in new_x:
        new_x.remove(att_name)
        new_x = [*new_x, *dummies.columns.values]
    
    return df_enc, new_x

Let's try our new function on the ``test`` dataset with the attribute 'Name'.

In [ ]:
test, x = dummies(test, x, 'Name')

In [ ]:
test.info()

__Now, apply the function on the other categorical attributes for both ``train`` and ``test`` datasets.__

Note: Do not do the change for the ID attribute, because it contains too many values, and we already decided to ignore it for prediction. In general, we don't have to do the change for the attributes that we ignore in the prediction.

Use the function ``dtypes`` to verify that you don't forget any categorical attribute.

In [ ]:
# Get dummies for train and test

train, x = dummies(train, x, '?')
test, x = dummies(test, x, '?')
# do the same for other attributes

In [ ]:
# Last check

train.info()

In [ ]:
test.info()

In [ ]:
x

### 3.4 Apply the algorithm (5 min) <a class="anchor" id="t34"></a>

To apply a ML algorithm, we first need to build the model. Here, we will use the algorithm ``KNeighborsRegressor`` which is an algorithm from the library ``sklearn``. It is a KNN algorithm for a regression problem. In our case, we want to predict the ETT, a numerical feature.

The ``KNeighborsRegressor`` takes as an parameter the number of neighbors. In this exercise, we do not focus on tuning the model, so we will take a value of 25.

In [ ]:
# Import and build the model

from sklearn.neighbors import KNeighborsRegressor
knn = KNeighborsRegressor(n_neighbors = 25)

Once the model is built, we want to train it on our training dataset. For that, we use the function ``fit()`` from the model itself.

Most algorithms have usually a function ``fit()`` or a similar one. Always check the official documentation if you are unsure on how to use the model (for example, here is the documentation for the ``KNeighborsRegressor``: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html ).

Here, the function ``fit()`` takes as argument the training data without the label (``x``) and the label (``y``).

In [ ]:
# Train the model

knn.fit(train[x], train[y])

Once the model is trained, we can generate the predictions on our test set. For that, we use here the function ``predict()`` of the model. This function takes as an argument the test set without the labels, and returns a series of predictions.

In [ ]:
# Get the predictions

predictions = knn.predict(test[x])

## 4 Measure the performance (5 min) <a class="anchor" id="t4"></a>

We can examine the variable ``predictions``, and see that it looks the same as the series or real labels that we have in the variable ``test[y]``.

In [ ]:
predictions

In [ ]:
test[y]

There are many ways to measure the performance of a model. In the case of a regression problem, we can simply compute the mean absolute error with the function ``mean_absolute_error()`` from the library ``sklearn``.

The mean absolute error (MAE) represents the average difference between the prediction and the real value. It is given in the unit of the predicted attribute. The RAE (relative absolute error) is the same but expressed in percent on the dataset (divided by the variation of the predicted attribute).

We can choose one metric or another to express the performance, each metrics will give us a different understanding on the error.

In [ ]:
# Import and compute the mean absolute error

from sklearn.metrics import mean_absolute_error
mean_absolute_error(predictions, test[y])

This error gives us the average error in minutes of our model, for the prediction of the 'ETT'.

## 5 Summary questions <a class="anchor" id="t5"></a>

*Which type of attribute cannot be processed raw by the KNN algorithm?*

*Which numeric attributes should be transformed to better represent the reality?*

*What can we say about the points which have a wrong value for the 3 attributes mentioned? Should we discard these points, or recover the values? Why?*

*Which attributes are useless for the prediction, and why?*

*Which other attribute is not detected by the usual numeric methods, but in our case does not provide any additional information for the prediction?*

*How can we simplify the model without any loss of information from the attributes with the same distribution?*